# pICNN → NGSolve : Phase 1 Proof of Concept

Goal (committee's request): establish the **PyTorch ↔ NGSolve UMAT bridge** with a *simple* network that reproduces compressible Neo-Hookean, and prove it works via:
- **V3** exact autograd tangent vs. central difference
- **V4** quadratic Newton convergence on a single element driven by the NN material

Run cells top to bottom. Cells 1–9 need only `torch`/`numpy`; the NGSolve bridge (cell 10) needs `ngsolve`.

```bash
pip install torch numpy matplotlib ngsolve
```

In [ ]:
import torch, numpy as np
import matplotlib.pyplot as plt
from torch.func import jacrev, vmap
import torch.nn as nn, torch.nn.functional as Fn

torch.set_default_dtype(torch.float64)   # double precision -> clean tangent checks
dev = 'cpu'
print('torch', torch.__version__)

## 1. Reference model & Phase-1 data
A single element under affine displacement has a *homogeneous* F, so the 'homogenized' response equals the pointwise material response — no averaging needed. So the Phase-1 dataset is analytic Neo-Hookean sampled over admissible F.

In [ ]:
MU, LAM = 1.0, 10.0

def psi_nh(F, mu=MU, lam=LAM):
    C = F.transpose(-1,-2) @ F
    I1 = C.diagonal(dim1=-2, dim2=-1).sum(-1)
    J = torch.linalg.det(F); lnJ = torch.log(J)
    return 0.5*mu*(I1-3.0) - mu*lnJ + 0.5*lam*lnJ**2

def P_nh(F, mu=MU, lam=LAM):
    FinvT = torch.linalg.inv(F).transpose(-1,-2)
    lnJ = torch.log(torch.linalg.det(F))[...,None,None]
    return mu*(F - FinvT) + lam*lnJ*FinvT

def make_dataset(n=20000, s=0.3):
    H = s*(torch.rand(n,3,3)-0.5)*2.0
    F = torch.eye(3) + H
    F = F[torch.linalg.det(F) > 0.2]
    return F, psi_nh(F), P_nh(F)

F_tr, psi_tr, P_tr = make_dataset()
print('dataset F:', F_tr.shape)

## 2. The (p)ICNN
Features are convex functions of the polyconvex args (I1, I2, J, -lnJ). All weights non-negative (softplus) + softplus activation ⇒ convex & non-decreasing ⇒ polyconvex ⇒ SPD tangent. Reference normalization removes both energy AND stress at F=I.

In [ ]:
def features(F):
    C = F.transpose(-1,-2) @ F
    I1 = C.diagonal(dim1=-2, dim2=-1).sum(-1)
    I2 = 0.5*(I1**2 - (C*C).sum((-1,-2)))
    J = torch.linalg.det(F)
    return torch.stack([I1, I2, J, -torch.log(J)], dim=-1)

class ICNN(nn.Module):
    def __init__(self, n_in=4, widths=(32,32,32)):
        super().__init__()
        dims=[n_in]+list(widths)
        self.Wy=nn.ParameterList([nn.Parameter(torch.randn(dims[l+1],n_in)*0.1) for l in range(len(widths))])
        self.Wz=nn.ParameterList([nn.Parameter(torch.randn(dims[l+1],dims[l])*0.1) for l in range(1,len(widths))])
        self.b =nn.ParameterList([nn.Parameter(torch.zeros(dims[l+1])) for l in range(len(widths))])
        self.out=nn.Parameter(torch.randn(widths[-1])*0.1); self.out_b=nn.Parameter(torch.zeros(()))
    def forward(self, y):
        z=None
        for l in range(len(self.Wy)):
            pre = Fn.linear(y, Fn.softplus(self.Wy[l])) + self.b[l]
            if z is not None: pre = pre + Fn.linear(z, Fn.softplus(self.Wz[l-1]))
            z = Fn.softplus(pre)
        return Fn.linear(z, Fn.softplus(self.out)) + self.out_b

class HyperelasticNN(nn.Module):
    def __init__(self):
        super().__init__(); self.net = ICNN()

    # ADD THIS REQUIRED PYTORCH METHOD
    def forward(self, F):
        return self.energy(F)
    
    def energy(self, F):
        I = torch.eye(3).expand_as(F).clone().requires_grad_(True)
        psi_I = self.net(features(I))
        P_I = torch.autograd.grad(psi_I.sum(), I, create_graph=True)[0]
        psi = self.net(features(F))
        return psi - psi_I.detach() - (P_I.detach()*(F - I.detach())).sum((-1,-2))

model = HyperelasticNN().to(dev)
print('params:', sum(p.numel() for p in model.parameters()))

## 3. Exact stress P and tangent A via autograd

In [ ]:
def stress_and_tangent(model, F):
    def e(f): return model.energy(f.unsqueeze(0)).squeeze()
    P = vmap(jacrev(e))(F)
    A = vmap(jacrev(jacrev(e)))(F)
    return P, A            # (n,3,3), (n,3,3,3,3)

## 4. Sobolev training (energy + stress)
Prioritise the stress term (drives Newton convergence).

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
lam_psi, lam_P = 1.0, 10.0
hist=[]
for ep in range(1500):
    idx = torch.randint(0, len(F_tr), (512,))
    F = F_tr[idx].clone()
    P_hat, _ = stress_and_tangent(model, F)
    psi_hat = model.energy(F)
    L = lam_psi*((psi_hat-psi_tr[idx])**2).mean() + lam_P*((P_hat-P_tr[idx])**2).mean()
    opt.zero_grad(); L.backward(); opt.step()
    hist.append(L.item())
    if ep%200==0: print(f'ep {ep:4d}  loss {L.item():.3e}')
plt.semilogy(hist); plt.xlabel('epoch'); plt.ylabel('loss'); plt.title('Sobolev training'); plt.show()

In [ ]:
# sanity: predicted vs analytic P over a uniaxial stretch path
lam = torch.linspace(0.8, 1.3, 40)
Fp = torch.eye(3).repeat(40,1,1).clone(); Fp[:,0,0]=lam
Pp,_ = stress_and_tangent(model, Fp)
plt.plot(lam, P_nh(Fp)[:,0,0], 'k-', label='analytic P11')
plt.plot(lam, Pp[:,0,0].detach(), 'r--', label='NN P11')
plt.legend(); plt.xlabel('stretch'); plt.ylabel('P11'); plt.show()
torch.save(model.state_dict(), "/home/vignesh/PPP/Project implementation plan (1)/phase1.pt")

## 5. V3 — autograd tangent vs central difference
Answers Prüger's slide-8 question: the FD error is V-shaped, bottoming ~1e-8 near eps=1e-6. Machine precision is unreachable by FD *by construction* — which is exactly why we use autograd.

In [ ]:
def cd_tangent(model, F, eps):
    n=F.shape[0]; A=torch.zeros(n,3,3,3,3)
    for k in range(3):
        for l in range(3):
            dF=torch.zeros_like(F); dF[:,k,l]=eps
            Pp,_=stress_and_tangent(model,F+dF); Pm,_=stress_and_tangent(model,F-dF)
            A[...,k,l]=(Pp-Pm)/(2*eps)
    return A

Fchk = torch.eye(3)+0.05*torch.randn(4,3,3)
_, A_ex = stress_and_tangent(model, Fchk)
eps_list=[1e-2,1e-4,1e-6,1e-8]; errs=[]
for e in eps_list:
    rel=((cd_tangent(model,Fchk,e)-A_ex).norm()/A_ex.norm()).item(); errs.append(rel)
    print(f'eps={e:.0e}  rel.err={rel:.2e}')
plt.loglog(eps_list, errs, 'o-'); plt.xlabel('eps'); plt.ylabel('rel err'); plt.title('V3'); plt.show()

## 6. V4 — NGSolve bridge: single element, NN-driven Newton
Uses `IntegrationRuleSpace` to extract F at integration points, evaluate the PyTorch model, and assemble residual/stiffness with the exact autograd tangent. Expect quadratic drop of |R|.

In [ ]:
from netgen.occ import Box, OCCGeometry, Pnt
from ngsolve import *

geo = OCCGeometry(Box(Pnt(0,0,0), Pnt(1,1,1)))
mesh = Mesh(geo.GenerateMesh(maxh=1.0))
mesh.ngmesh.SetBCName(0,'xmin')  # ensure named faces exist; adjust to your geometry
order=1
fes = VectorH1(mesh, order=order, dirichlet='.*')  # clamp all faces for a driven test
u, v = fes.TrialFunction(), fes.TestFunction()
gfu = GridFunction(fes)
gfu.Set(CF((0.1*x,0,0)))   # prescribed uniform stretch as Dirichlet data

irs   = IntegrationRuleSpace(mesh, order=2*order)
gP, gA, Fir = GridFunction(irs**9), GridFunction(irs**81), GridFunction(irs**9)
Pmat = CF(tuple(gP.components[i] for i in range(9)), dims=(3,3))
Amat = CF(tuple(gA.components[i] for i in range(81)), dims=(9,9))

res = LinearForm(fes); res += InnerProduct(Pmat, Grad(v))*dx
K   = BilinearForm(fes)
K  += InnerProduct(Amat*Grad(u).Reshape((9,)), Grad(v).Reshape((9,)))*dx

def update_material():
    Fir.Interpolate((Id(3)+Grad(gfu)).Reshape((9,)))
    Fnp = Fir.vec.FV().NumPy().reshape(-1,3,3)
    P,A = stress_and_tangent(model, torch.tensor(Fnp))
    gP.vec.FV().NumPy()[:] = P.reshape(-1,9).detach().numpy().ravel()
    gA.vec.FV().NumPy()[:] = A.reshape(-1,81).detach().numpy().ravel()

for it in range(12):
    update_material(); res.Assemble(); K.Assemble()
    r = res.vec.CreateVector(); r.data = -res.vec
    du = K.mat.Inverse(fes.FreeDofs()) * r
    gfu.vec.data += du
    print(f'Newton {it}: |R|={Norm(r):.3e}')
    if Norm(r) < 1e-10: break

## Next steps (Phase 2)
- Mixed u–p formulation for the near-incompressible PDMS matrix (Taylor–Hood) to kill volumetric locking.
- Periodic RVE: `Periodic()` on a periodic OCC mesh; rigid inclusions via rigid-body dof constraints (not E→∞).
- Hill–Mandel averaging + *consistent* (statically condensed) macroscopic tangent — no finite difference.
- Retrain the ICNN as a *partial* ICNN with volume fraction v_f as the non-convex context input.